# 📓 Notebook 20 — LangGraph---**Phase 7 · Industry Frameworks**> LangGraph is the **production standard** for building stateful, multi-step agent workflows as directed graphs. If LangChain is the framework, LangGraph is the orchestrator.### What You'll Learn| Topic | Why It Matters ||-------|---------------|| State graphs | Define agent workflows as graphs || Nodes & edges | Functions as nodes, routing as edges || Conditional routing | LLM-driven decision at branch points || Cycles & loops | ReAct, reflection loops as graph cycles || Checkpointing | Persist and resume agent state || Human-in-the-loop | Pause for human approval |### 🏗️ Build: ReAct Agent with LangGraph> **Prerequisite:** `pip install langgraph langchain-openai`

## 1. Setup

In [ ]:
import os, jsonfrom dotenv import load_dotenvload_dotenv()from langchain_openai import ChatOpenAIfrom langchain_google_genai import ChatGoogleGenerativeAIif os.getenv("OPENAI_API_KEY"):    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)    print("✅ Using OpenAI")elif os.getenv("GOOGLE_API_KEY"):    llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)    print("✅ Using Gemini")print(f"🤖 LLM: {llm.__class__.__name__}")

---## 2. LangGraph Core Concepts### Why LangGraph?| Problem with Simple Chains | LangGraph Solution ||---------------------------|--------------------|| Linear flow only | Arbitrary graph topology || No state between steps | Built-in state management || No cycles/loops | Supports cycles (ReAct, reflection) || Can't pause/resume | Checkpointing & persistence || No conditional branching | Conditional edges |### The Graph Model```       ┌─────────┐       │  START   │       └────┬─────┘            ↓       ┌─────────┐       ┌─────────┐       │  Agent   │──────→│  Tools  │       │  (node)  │←──────│  (node) │       └────┬─────┘       └─────────┘            ↓ (condition: no more tool calls)       ┌─────────┐       │   END    │       └──────────┘```- **Nodes** = Python functions that transform state- **Edges** = Connections between nodes (can be conditional)- **State** = TypedDict that flows through the graph

---## 3. Building Your First Graph

In [ ]:
from typing import TypedDict, Annotated, Sequencefrom langchain_core.messages import HumanMessage, AIMessage, BaseMessagefrom langgraph.graph import StateGraph, END, STARTfrom langgraph.graph.message import add_messages# Step 1: Define the stateclass AgentState(TypedDict):    messages: Annotated[Sequence[BaseMessage], add_messages]# Step 2: Define node functionsdef chatbot_node(state: AgentState):    """Call the LLM with current messages."""    response = llm.invoke(state["messages"])    return {"messages": [response]}# Step 3: Build the graphgraph = StateGraph(AgentState)# Add nodesgraph.add_node("chatbot", chatbot_node)# Add edgesgraph.add_edge(START, "chatbot")graph.add_edge("chatbot", END)# Compilesimple_chain = graph.compile()# Testresult = simple_chain.invoke({"messages": [HumanMessage(content="What is LangGraph in one sentence?")]})print(f"🤖 {result['messages'][-1].content}")

In [ ]:
# Visualize the graph (if in Jupyter with display support)try:    from IPython.display import Image, display    display(Image(simple_chain.get_graph().draw_mermaid_png()))except:    print("📊 Graph structure:")    print(simple_chain.get_graph().draw_mermaid())

---## 4. ReAct Agent with LangGraphThis is the canonical agent pattern — the most important LangGraph pattern to know.

In [ ]:
from langchain_core.tools import toolfrom langgraph.prebuilt import create_react_agentimport math# Define tools@tooldef calculator(expression: str) -> str:    """Evaluate a mathematical expression. Supports +, -, *, /, **, sqrt, sin, cos, log, pi."""    try:        safe = {"sqrt": math.sqrt, "sin": math.sin, "cos": math.cos,                 "pi": math.pi, "e": math.e, "abs": abs, "log": math.log}        result = eval(expression.replace("^","**"), {"__builtins__": {}}, safe)        return f"Result: {result}"    except Exception as e:        return f"Error: {e}"@tooldef search_knowledge(query: str) -> str:    """Search for factual information about a topic."""    knowledge = {        "python": "Python is a high-level language created in 1991. It's the most popular language for ML/AI.",        "langchain": "LangChain is the most popular framework for building LLM applications, supporting chains, agents, RAG.",        "langgraph": "LangGraph is a library for building stateful, multi-actor agent workflows as graphs. Built on LangChain.",        "transformer": "Transformer architecture uses self-attention. Introduced in 'Attention Is All You Need' (2017).",        "rag": "RAG combines retrieval and generation to ground LLM answers in external knowledge, reducing hallucination.",    }    q = query.lower()    for key, info in knowledge.items():        if key in q:            return info    return f"Information about '{query}' not found in knowledge base."tools = [calculator, search_knowledge]# Create ReAct agent using LangGraph's prebuiltreact_agent = create_react_agent(    llm,    tools,    prompt="You are a helpful research assistant. Use tools when needed for accurate answers.")# Testprint("🤖 LangGraph ReAct Agent")print("=" * 60)queries = [    "What is LangGraph and how does it relate to LangChain?",    "Calculate the circumference of a circle with radius 15.7",    "What is RAG? Then calculate how many tokens a 1000-word document might have (assume 1.3 tokens per word)",]for q in queries:    print(f"\n❓ {q}")    result = react_agent.invoke({"messages": [HumanMessage(content=q)]})    print(f"📝 {result['messages'][-1].content}")

---## 5. Custom Multi-Step Agent Graph

In [ ]:
from langgraph.graph import StateGraph, START, ENDfrom langchain_core.messages import SystemMessageclass ResearchState(TypedDict):    messages: Annotated[Sequence[BaseMessage], add_messages]    research_notes: str    quality_score: intdef researcher_node(state: ResearchState):    """Research the topic."""    msgs = [        SystemMessage(content="You are a thorough researcher. Provide detailed, factual information."),        *state["messages"]    ]    response = llm.invoke(msgs)    return {"messages": [response], "research_notes": response.content}def reviewer_node(state: ResearchState):    """Review and score the research."""    review_prompt = f"""Rate this research 1-10 for completeness and accuracy:{state['research_notes']}Respond with just the number."""    response = llm.invoke([HumanMessage(content=review_prompt)])    try:        score = int(''.join(filter(str.isdigit, response.content[:3])))    except:        score = 5        feedback = f"Review score: {score}/10"    return {"messages": [AIMessage(content=feedback)], "quality_score": score}def improver_node(state: ResearchState):    """Improve research based on review."""    improve_prompt = f"""The research below scored {state['quality_score']}/10. Improve it significantly:{state['research_notes']}Provide an enhanced, more detailed version."""    response = llm.invoke([HumanMessage(content=improve_prompt)])    return {"messages": [response], "research_notes": response.content}# Conditional edge: should we improve or finalize?def should_improve(state: ResearchState):    if state.get("quality_score", 0) >= 8:        return "end"    return "improve"# Build the graphresearch_graph = StateGraph(ResearchState)research_graph.add_node("research", researcher_node)research_graph.add_node("review", reviewer_node)research_graph.add_node("improve", improver_node)research_graph.add_edge(START, "research")research_graph.add_edge("research", "review")research_graph.add_conditional_edges("review", should_improve, {"improve": "improve", "end": END})research_graph.add_edge("improve", "review")  # Loop back for re-reviewresearch_chain = research_graph.compile()# Visualizetry:    from IPython.display import Image, display    display(Image(research_chain.get_graph().draw_mermaid_png()))except:    print("📊 Graph:")    print(research_chain.get_graph().draw_mermaid())

In [ ]:
# Run the research graphprint("🔬 Research Graph Demo")print("=" * 60)result = research_chain.invoke({    "messages": [HumanMessage(content="Explain how transformers work in deep learning")],    "research_notes": "",    "quality_score": 0,})print(f"\n📝 Final Research ({len(result['research_notes'])} chars):")print(f"   Score: {result['quality_score']}/10")print(f"   Rounds: {len(result['messages'])} messages")print(f"\n{result['research_notes'][:500]}...")

---## 6. Checkpointing — Persist and Resume

In [ ]:
from langgraph.checkpoint.memory import MemorySaver# Add checkpointing to any graphcheckpointer = MemorySaver()# Recompile with checkpointerpersistent_agent = create_react_agent(    llm, tools,    prompt="You are a helpful assistant that remembers conversation context.",    checkpointer=checkpointer,)# Use thread_id to maintain separate conversationsconfig = {"configurable": {"thread_id": "user_abhishek_session_1"}}# Multi-turn conversation with persistenceturns = [    "Hi, I'm researching RAG systems. What is RAG?",    "What tools can I use to build one?",    "Earlier you mentioned RAG reduces something — what was it?",]print("💾 Checkpointed Conversation")print("=" * 60)for turn in turns:    result = persistent_agent.invoke(        {"messages": [HumanMessage(content=turn)]},        config=config,    )    print(f"\n👤 {turn}")    print(f"🤖 {result['messages'][-1].content[:200]}...")

---## 7. Human-in-the-Loop

In [ ]:
from langgraph.prebuilt import create_react_agent# Create agent that requires human approval for certain actions@tooldef delete_file(filename: str) -> str:    """Delete a file from the system. THIS IS A DESTRUCTIVE ACTION."""    return f"File '{filename}' deleted successfully."@tooldef read_file(filename: str) -> str:    """Read contents of a file. Safe, read-only operation."""    return f"Contents of {filename}: [sample data]"# With interrupt_before, the graph pauses before executing the tools nodehitl_agent = create_react_agent(    llm,    [read_file, delete_file],    checkpointer=MemorySaver(),    interrupt_before=["tools"],  # Pause before tool execution)config = {"configurable": {"thread_id": "approval_demo"}}# This will pause before tool executionprint("🛑 Human-in-the-Loop Demo")print("=" * 60)print("Agent will pause before executing tools for human approval.\n")result = hitl_agent.invoke(    {"messages": [HumanMessage(content="Read the file config.yaml")]},    config=config,)# Check current statesnapshot = hitl_agent.get_state(config)print(f"Status: {'interrupted' if snapshot.next else 'complete'}")if snapshot.next:    print(f"Waiting at: {snapshot.next}")    # In production: show pending tool calls to human for approval    pending = result["messages"][-1]    if hasattr(pending, "tool_calls"):        for tc in pending.tool_calls:            print(f"  Pending tool: {tc['name']}({tc['args']})")        # Approve by resuming (pass None to continue)    print("\n✅ Approving tool execution...")    final = hitl_agent.invoke(None, config=config)    print(f"🤖 {final['messages'][-1].content}")

---## 📝 Interview Quiz — LangGraph### Q1: Why use LangGraph instead of plain LangChain agents?<details><summary>💡 Answer</summary>LangChain's `AgentExecutor` is a simple loop. LangGraph provides:1. **Explicit control flow** — You define exactly how the agent moves between steps2. **State management** — Typed state that flows through the graph3. **Cycles** — Support for loops (ReAct, reflection) in a controlled way4. **Checkpointing** — Persist state to resume later5. **Human-in-the-loop** — Pause at any node for human approval6. **Multi-agent** — Multiple agents as separate nodes in one graph7. **Streaming** — Token-level streaming of any node's output**When to use LangGraph:** Any agent workflow that needs persistence, approval gates, multi-agent coordination, or complex control flow.</details>### Q2: What is a conditional edge in LangGraph?<details><summary>💡 Answer</summary>A **conditional edge** is a routing function that determines which node to go to next based on the current state.```pythondef should_improve(state):    if state["quality_score"] >= 8:        return "end"    return "improve"graph.add_conditional_edges("review", should_improve, {    "improve": "improve_node",    "end": END})```This enables dynamic branching — the LLM's output determines the next step. Essential for ReAct (tool calls vs final answer) and reflection (improve vs accept).</details>### Q3: What is checkpointing and why does it matter for production?<details><summary>💡 Answer</summary>**Checkpointing** saves the complete graph state at each step.**Benefits:**- **Resume** — If agent fails, resume from last checkpoint (not restart)- **Multi-turn** — Maintain conversation state across API calls- **Debugging** — Inspect state at any step for troubleshooting- **Human-in-the-loop** — Pause, wait for human input, resume- **Time travel** — Replay and branch from any past state**Backends:** MemorySaver (dev), SqliteSaver (single-instance), PostgresSaver (production).</details>### Q4: How would you build a multi-agent system with LangGraph?<details><summary>💡 Answer</summary>Each agent is a **node** in the graph:```pythongraph.add_node("researcher", researcher_node)graph.add_node("writer", writer_node)graph.add_node("reviewer", reviewer_node)graph.add_edge("researcher", "writer")graph.add_edge("writer", "reviewer")graph.add_conditional_edges("reviewer", decide_next, {    "revise": "writer",    # Loop back    "done": END            # Accept})```Agents share state through the `TypedDict`. Each node reads from and writes to the shared state. This is the **supervisor pattern** — one graph controls multiple specialized agents.</details>

---## ✅ Notebook 20 Summary| Concept | Key Takeaway ||---------|-------------|| StateGraph | Define agent workflow as nodes + edges + state || Nodes | Python functions that transform state || Conditional edges | LLM-driven routing decisions || Cycles | ReAct, reflection loops as graph cycles || Checkpointing | Persist state for resume, multi-turn, debugging || Human-in-the-loop | `interrupt_before` pauses for human approval || Prebuilt agents | `create_react_agent` for quick ReAct setup |### ➡️ Next: [Notebook 21 — LangSmith](./21_langsmith.ipynb)